**Task1:** Conversational history with summarization

 This notebook manages user–assistant chat history and generates summaries after a fixed number of conversation turns (k-th run).

In [ ]:
import os
import json
from jsonschema import validate

# importing all the necessary libraries and setting up the groq api key

os.environ["GROQ_API_KEY"] = "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"

# Initializing the client object in openAI instance

from openai import OpenAI
client = OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")

In [26]:
class ConversationManager:
    def __init__(self, max_turns=10, max_chars=1000, summarize_every=3):

        # k: Generate summary after every k messages
        # max_length: Maximum allowed character length of history

        self.history = []
        self.summaries = []
        self.max_turns = max_turns
        self.max_chars = max_chars
        self.summarize_every = summarize_every
        self.turn_count = 0

    def add_message(self, role, content):

        # Add a message to conversation history
        # role: "user" or "assistant"
        # content: text of the message"

        self.history.append({"role": role, "content": content})
        self.turn_count += 1

        # Enforce history length limit

        self.truncate_history()

        # Summarize history after every k-th message

        if self.turn_count % self.summarize_every == 0:
            self.create_summary()

    def truncate_history(self):
        if len(self.history) > self.max_turns:
            self.history = self.history[-self.max_turns:]
        while sum(len(m["content"]) for m in self.history) > self.max_chars:
            self.history.pop(0)

    def create_summary(self, last_n_messages=20):
      msgs = self.history[-last_n_messages:]
      convo_text = "\n".join([f"{m['role'].capitalize()}: {m['content']}" for m in msgs])

      system_prompt = (
        "You are a concise and factual summarization assistant. "
        "Given the conversation below, produce a short summary (2-4 sentences) that includes:\n"
        "1) Main topic(s) discussed\n"
        "2) The user's main concern(s) or question(s)\n"
        "3) The assistant's recommendations or action items (if any)\n\n"
        "IMPORTANT: Only use facts present in the conversation. Do NOT invent details. "
        "If something is not specified, do not guess — say 'unclear' for that part. "
        "Return ONLY the summary text (no headings, no extra explanation)."
      )

      try:
          response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": convo_text}
            ],
            temperature=0.0,
            max_tokens=200
          )

          # Extracting summary text from the model output.
          summary_text = response.choices[0].message.content.strip()

          # Handling cases where responses are empty
          if not summary_text:
            raise ValueError("Empty summary returned")

          # Saving summaries with counting the turns
          self.summaries.append(f"[Summary after {self.turn_count} turns]: {summary_text}")

      except Exception as e:
        # A fallback , used in case summarization fails , just use first 60 words of the conversation text
        fallback = " ".join(convo_text.split()[:60])
        self.summaries.append(f"[Summary fallback after {self.turn_count} turns]: {fallback}...")
        print("⚠️ Summarization failed — used fallback. Error:", e)

    # Getting the full convo history
    def get_history(self):
        return self.history

    # Returning all the stored summaries
    def get_summaries(self):
        return self.summaries



In [27]:
# This is the demonstration cell where we pass the convo to the LLM via manager-> ConversationManager


def pretty_print_history(history):
    print("=== Conversation History ===")
    for msg in history:
        role = msg["role"].capitalize()
        content = msg["content"]
        print(f"{role}: {content}")
    print("="*40)

def pretty_print_summaries(summaries):
    print("=== Summaries ===")
    for s in summaries:
        print(s)
    print("="*40)

manager = ConversationManager(max_turns=5, max_chars=300, summarize_every=3)
manager.add_message("user", "I’ve been feeling tired lately.")
manager.add_message("assistant", "Are you sleeping enough and eating balanced meals?")
manager.add_message("user", "I sleep only 5 hours on weekdays.")
manager.add_message("assistant", "That could be a factor. Aim for 7–8 hours.")
manager.add_message("user", "What about diet? I mostly eat fast food.")
manager.add_message("assistant", "Try including fruits, vegetables, and lean protein.")
manager.add_message("user", "Should I take supplements too?")
manager.add_message("assistant", "Start with fixing sleep and diet first, then consult a doctor if fatigue continues.")

pretty_print_history(manager.get_history())
pretty_print_summaries(manager.get_summaries())




=== Conversation History ===
Assistant: That could be a factor. Aim for 7–8 hours.
User: What about diet? I mostly eat fast food.
Assistant: Try including fruits, vegetables, and lean protein.
User: Should I take supplements too?
Assistant: Start with fixing sleep and diet first, then consult a doctor if fatigue continues.
=== Summaries ===
[Summary after 3 turns]: The main topic discussed is the user's fatigue. The user's main concern is their sleep schedule, specifically sleeping only 5 hours on weekdays. The assistant's recommendation is unclear, as the conversation does not provide further guidance.
[Summary after 6 turns]: The main topic discussed is sleep and diet. The user's main concern is their sleep schedule and diet, specifically eating mostly fast food. The assistant's recommendations are to aim for 7-8 hours of sleep and include fruits, vegetables, and lean protein in their diet.
